# S3 J6 — Context Engineering

Notebook étudiant généré à partir des sources Markdown du jour.

## Objectifs

- Construire un context pack.
- Respecter un budget.
- Filtrer par visibilité.
- Prévenir les fuites de contexte.

# Chapitre — Context Engineering

## 1. Pourquoi le Context Engineering existe

Un agent IA ne raisonne jamais dans le vide. Il reçoit une entrée composée de plusieurs couches :

- instructions système ;
- message utilisateur courant ;
- historique conversationnel ;
- état de tâche ;
- mémoire utilisateur ;
- résultats de recherche ;
- ressources MCP ;
- schémas d’outils ;
- traces ou décisions intermédiaires.

Le problème est que toutes ces informations ne doivent pas être injectées au modèle à chaque tour.

Le **Context Engineering** est la discipline qui consiste à concevoir, sélectionner et organiser le contexte disponible pour qu’un modèle puisse accomplir une tâche avec le moins de bruit possible.

## 2. Prompt Engineering vs Context Engineering

Le prompt engineering se concentre souvent sur la formulation d’une instruction.

Le context engineering se concentre sur le système complet qui décide :

- quelles informations sont disponibles ;
- lesquelles sont pertinentes ;
- lesquelles sont autorisées ;
- dans quel ordre elles doivent apparaître ;
- sous quelle forme elles doivent être fournies ;
- lesquelles doivent être exclues.

Un prompt peut être parfait et échouer si le contexte est mauvais.

Un contexte bien conçu permet au modèle de :

- comprendre l’objectif ;
- utiliser le bon état ;
- éviter les contradictions ;
- appeler les bons outils ;
- ignorer les informations obsolètes ;
- limiter les hallucinations.

## 3. Les couches de contexte

Dans un système agentique moderne, on distingue plusieurs couches.

### 3.1 Instruction système

Elle définit le rôle, les règles et les limites générales.

Exemple :

```text
Tu es un agent support interne. Tu dois répondre uniquement avec les sources autorisées.
```

Cette couche doit être stable et courte.

### 3.2 Objectif courant

L’objectif courant vient de la demande utilisateur ou du plan de l’agent.

Exemple :

```text
Résoudre un incident de facturation pour le client ACME.
```

Cette couche oriente la sélection.

### 3.3 État de tâche

L’état de tâche décrit la progression actuelle.

Exemple :

```json
{
  "ticket_id": "T-204",
  "status": "waiting_for_user",
  "missing_fields": ["invoice_id"]
}
```

Cette donnée est applicative. Elle ne doit pas être confondue avec un souvenir long terme.

### 3.4 Historique récent

L’historique récent contient les derniers tours de conversation utiles.

Il ne doit pas être conservé indéfiniment dans le prompt.

Un bon système conserve :

- les derniers messages importants ;
- les décisions actives ;
- les questions non résolues ;
- les contraintes explicitement données.

### 3.5 Mémoire longue

La mémoire longue conserve des préférences ou informations persistantes.

Exemple :

```text
L’utilisateur préfère des réponses avec exemples Python.
```

Une mémoire longue ne doit pas être injectée automatiquement. Elle doit passer par une politique de pertinence et d’autorisation.

### 3.6 Ressources MCP

Les ressources MCP peuvent fournir de la documentation, des fichiers, des entrées métier ou des données externes.

Elles doivent être sélectionnées en fonction :

- de l’objectif ;
- de leur fraîcheur ;
- de leur autorisation ;
- de leur granularité ;
- de leur coût contextuel.

### 3.7 Outils

Les outils disponibles influencent le comportement du modèle.

Inclure trop d’outils peut dégrader la sélection d’action. Un agent doit recevoir uniquement les outils utiles à la tâche courante.

## 4. Architecture générale

```mermaid
flowchart TD
    A[User request] --> B[Goal extractor]
    B --> C[Context inventory]
    C --> D[Policy filters]
    D --> E[Priority scoring]
    E --> F[Budget allocator]
    F --> G[Context pack]
    G --> H[Model call]
    H --> I[Action or answer]
    I --> J[Trace]
```

Le moteur de contexte agit avant l’appel modèle.

Il ne remplace pas l’agent. Il prépare son environnement informationnel.

## 5. Budget de contexte

La fenêtre de contexte d’un modèle est limitée.

Même quand elle est grande, tout injecter e

_Le chapitre complet est disponible dans `book/week03/day06/chapter.md`._

## Lab

Copiez ou importez le fichier `context_engineering.py`, puis exécutez la démonstration.

In [ ]:
# Depuis la racine du dépôt:
# python book/week03/day06/labs/context_engineering.py
# python book/week03/day06/labs/test_context_engineering.py

In [ ]:
from pathlib import Path
lab_path = Path("book/week03/day06/labs/context_engineering.py")
print(lab_path)

## Exercices

Voir `book/week03/day06/exercises.md`.

Le notebook étudiant ne contient pas les corrigés.

## Challenge

Construire un Context Builder par agent cible. Voir `challenge.md`.

# Corrigés — formateur

# Corrigé — Exercices — Jour 6

## Exercice 1 — Identifier les couches de contexte

| Élément | Catégorie |
|---|---|
| `Tu es un agent support interne.` | instruction système |
| `ticket_id = T-204` | état de tâche |
| `L’utilisateur préfère les réponses courtes.` | mémoire longue |
| Dernier message utilisateur | historique récent |
| `billing_policy.md` | ressource |
| `get_invoice_status` | outil |
| Tentative précédente d’outil | trace |

## Exercice 2 — Construire un context pack

À injecter :

- A — instruction système support ;
- C — état courant ;
- D — mémoire de préférence, si elle est autorisée et utile au format de réponse ;
- E — ressource de politique de relance ;
- F — outil de statut facture.

À exclure :

- B — ancien ticket RH, hors sujet ;
- G — `delete_invoice`, action dangereuse et inutile ;
- H — trace ancienne non liée.

## Exercice 3 — Budget

Budget : 100 tokens.

Sélection :

| Élément | Tokens |
|---|---:|
| system | 20 |
| task_state | 30 |

Total : 50.

`billing_policy` ferait passer le total à 120. Il doit être compressé, résumé ou récupéré partiellement.

`user_preference` peut être ajoutée si elle est utile, total 65.

`old_trace` est exclue car priorité faible.

## Exercice 4 — Visibilité

Un élément `private` peut contenir :

- raisonnement interne ;
- informations sensibles ;
- données destinées à un agent spécialisé ;
- diagnostics non validés ;
- secrets opérationnels.

Il ne doit pas être injecté sans transformation explicite, car cela peut créer une fuite d’information et influencer un agent qui n’a pas besoin de cette donnée.

## Exercice 5 — Déduplication

Les deux phrases portent la même information. Une déduplication normalisée supprime :

- espaces inutiles ;
- différences de casse ;
- variations mineures de format.

Elles doivent donc être considérées comme doublons.

## Exercice 6 — PII

Résultat attendu :

```text
Contacte Alice à [REDACTED_EMAIL] ou au [REDACTED_PHONE].
```

## Exercice 7 — MCP

Un serveur MCP peut exposer beaucoup de ressources. Les injecter toutes crée :

- bruit ;
- coût ;
- risque de fuite ;
- conflit de sources ;
- confusion d’outil ;
- latence.

Le client doit découvrir les ressources, puis sélectionner seulement celles qui répondent à l’objectif courant.

## Exercice 8 — Tests indispensables

Exemples de tests :

1. le moteur respecte le budget ;
2. les éléments `private` sont exclus si non autorisés ;
3. les emails et téléphones sont réduits ;
4. les doublons normalisés sont supprimés ;
5. les éléments critiques sont prioritaires ;
6. les raisons de rejet sont tracées.

# Corrigé — Questions d’entretien — Jour 6

## Question 1

Le Prompt Engineering concerne la formulation des instructions. Le Context Engineering concerne la construction du contexte complet transmis au modèle : sources, état, mémoire, outils, ressources, budget, permissions et ordre.

## Question 2

Envoyer tout l’historique augmente le coût, la latence, le bruit et les risques de fuite. Cela peut aussi introduire des instructions obsolètes ou contradictoires.

## Question 3

Je sélectionne les outils selon l’objectif courant, les permissions, la criticité de l’action, le domaine de l’agent et le risque opérationnel. Un agent ne doit voir que les outils utiles à la tâche.

## Question 4

La memory est une connaissance persistante. Le state décrit la progression applicative courante. Le context est le sous-ensemble réellement injecté dans l’appel modèle.

## Question 5

Je définis un budget, j’estime le coût de chaque élément, je trie par priorité, je préserve les éléments critiques, puis je supprime, résume ou récupère partiellement les éléments moins importants.

## Question 6

Un context pack est un paquet structuré contenant les éléments sélectionnés, les éléments rejetés, les raisons de rejet, le coût estimé et le rendu final pour modèle.

## Question 7

MCP standardise l’exposition d’outils, ressources et prompts. Le Context Engineering décide lesquels de ces éléments doivent réellement être utilisés ou injectés.

## Question 8

J’utilise des niveaux de visibilité, des politiques par agent, des filtres explicites, des audits et des tests empêchant l’injection non autorisée.

## Question 9

Les signaux utiles incluent fraîcheur, source, score de pertinence, priorité métier, taille, type, confiance, agent cible et lien avec l’objectif.

## Question 10

Je conserve les IDs des éléments sélectionnés, les éléments rejetés, les raisons de rejet, le budget, l’agent cible, la politique utilisée et le rendu final haché ou versionné.

# Corrigé — Challenge — Jour 6

## Implémentation de référence

```python
def build_for_agent(agent_name, items, policy):
    selected = []
    dropped = []
    total_tokens = 0

    for item in items:
        targets = item.get("target_agents", [])
        if targets and agent_name not in targets:
            dropped.append({"id": item["id"], "reason": "wrong_target_agent"})
            continue

        if item.get("visibility") == "private" and agent_name not in item.get("target_agents", []):
            dropped.append({"id": item["id"], "reason": "private_not_allowed"})
            continue

        tokens = item.get("tokens", len(item.get("content", "").split()))
        if total_tokens + tokens > policy["max_tokens"]:
            dropped.append({"id": item["id"], "reason": "budget_exceeded"})
            continue

        selected.append(item)
        total_tokens += tokens

    return {
        "agent": agent_name,
        "selected": selected,
        "dropped": dropped,
        "total_tokens": total_tokens,
    }
```

## Exemple `billing_agent`

Le `billing_agent` doit recevoir :

- son instruction système ;
- l’état de facture ;
- la politique de facturation ;
- les outils de lecture facture.

Il ne doit pas recevoir :

- notes privées sécurité ;
- outils destructifs non nécessaires ;
- traces internes du router.

## Exemple `security_agent`

Le `security_agent` peut recevoir :

- note de sécurité ;
- extrait du ticket ;
- règles de traitement des secrets ;
- outils de validation ou d’escalade.

Il ne doit pas recevoir automatiquement :

- détails de facturation inutiles ;
- préférences utilisateur sans rapport ;
- historique non lié.

## Tests attendus

1. `billing_agent` ne reçoit pas `security_note`.
2. `security_agent` reçoit `security_note` si elle lui est explicitement destinée.
3. Le budget est respecté.
4. Les éléments hors cible sont tracés.
5. Les emails sont réduits.
6. Le rendu final contient les IDs de sources.

## Réponse à la question de réflexion

Le contexte du `security_agent` peut contenir des hypothèses, secrets, alertes ou diagnostics qui ne sont pas utiles au traitement facturation. Le rendre visible au `billing_agent` peut exposer des informations sensibles et biaiser la réponse métier.

# Review formateur — Jour 6 — Context Engineering

## Résumé pédagogique

Cette journée fait passer les apprenants d’une vision simple du prompt à une vision système du contexte.

Point clé :

```text
Le contexte est un artefact d’architecture, pas une concaténation de messages.
```

## Concepts à vérifier

L’apprenant doit savoir expliquer :

- différence entre memory, state et context ;
- rôle du budget ;
- sélection d’outils ;
- ressources MCP ;
- visibilité ;
- PII ;
- déduplication ;
- trace des décisions.

## Déroulé conseillé

1. Revenir sur le jour 5 : état partagé.
2. Montrer qu’un état partagé n’est pas automatiquement un contexte modèle.
3. Présenter le concept de context pack.
4. Faire les exercices 1 à 3.
5. Exécuter le lab.
6. Modifier le budget pour observer les rejets.
7. Ajouter un élément privé et vérifier qu’il ne fuite pas.
8. Faire le challenge en binôme.

## Erreurs fréquentes

- Confondre mémoire longue et contexte.
- Injecter tout l’historique.
- Donner tous les outils à tous les agents.
- Oublier les raisons de rejet.
- Ne pas tester les cas de fuite.
- Penser uniquement en tokens et pas en permissions.

## Questions de relance

- Quel contexte supprimerais-tu en premier ?
- Que faire si un document critique dépasse le budget ?
- Comment auditer la sélection ?
- Pourquoi un outil disponible n’est-il pas toujours un outil exposé ?
- Que doit voir un reviewer que le coder ne voit pas ?

## Critères de validation

Le lab est validé si :

- les tests passent ;
- le budget est respecté ;
- la visibilité est appliquée ;
- les doublons sont supprimés ;
- la PII est réduite ;
- le rendu final est lisible ;
- les rejets sont traçables.

## Propositions d’amélioration

Ces propositions ne modifient pas les spécifications figées.

- Ajouter une version avancée avec scoring sémantique.
- Ajouter un tokenizer réel.
- Ajouter un exemple MCP avec ressources distantes.
- Ajouter une visualisation du budget par couche.